In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

zip_path = "/content/drive/MyDrive/garbage classification p.../dataset/archive.zip"

print(zip_path)
print(os.path.exists(zip_path))

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/garbage classification project"))


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/garbage classification project/dataset"))

In [ ]:
import os

# Create the destination directory if it doesn't exist
output_dir = "/content/dataset"
os.makedirs(output_dir, exist_ok=True)

# Unzip the archive.zip from Google Drive to the specified output directory
!unzip "/content/drive/MyDrive/garbage classification project/dataset/archive.zip" -d "{output_dir}"

In [ ]:
import os

print(os.listdir("/content/dataset"))


In [ ]:
import os

path = "/content/dataset/Garbage classification/Garbage classification"

print(os.listdir(path))

In [ ]:
dataset_path = "/content/dataset/Garbage classification/Garbage classification"

print(os.listdir(dataset_path))

In [ ]:
import os, shutil, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json (from kaggle.com/settings -> API -> Create New Token)

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d asdasdasasdas/garbage-classification
!unzip -q garbage-classification.zip -d garbage_data

In [ ]:
DATA_DIR = None
for root, dirs, files_ in os.walk("garbage_data"):
    if len(dirs) >= 5:  # the folder containing the 6 class subfolders
        DATA_DIR = root
        break

print("Detected data dir:", DATA_DIR)
print("Classes found:", os.listdir(DATA_DIR))

In [ ]:
BASE_DIR = "garbage_split"
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)

classes = [c for c in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, c))]
val_ratio = 0.2

for cls in classes:
    src = os.path.join(DATA_DIR, cls)
    imgs = os.listdir(src)
    random.shuffle(imgs)
    split = int(len(imgs) * (1 - val_ratio))
    for phase, subset in [("train", imgs[:split]), ("val", imgs[split:])]:
        dst = os.path.join(BASE_DIR, phase, cls)
        os.makedirs(dst, exist_ok=True)
        for img in subset:
            shutil.copy(os.path.join(src, img), os.path.join(dst, img))

print("Classes:", classes)
print("Train counts:", {c: len(os.listdir(f"{BASE_DIR}/train/{c}")) for c in classes})
print("Val counts:", {c: len(os.listdir(f"{BASE_DIR}/val/{c}")) for c in classes})ASE_DIR = "garbage_split"

In [ ]:
IMG_SIZE = (300, 300)   # EfficientNetB3's native input size
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    f"{BASE_DIR}/train", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True
)
val_gen = val_datagen.flow_from_directory(
    f"{BASE_DIR}/val", target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

NUM_CLASSES = train_gen.num_classes
print(train_gen.class_indices)

In [ ]:
base_model = EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False  # freeze for stage 1

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)
model.summary()

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("best_stage1.keras", monitor="val_accuracy", save_best_only=True),
]

In [ ]:
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
base_model.trainable = True
# Freeze the first ~70% of layers, fine-tune the rest
fine_tune_at = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),  # much smaller LR for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_ft = [
    EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint("best_finetuned.keras", monitor="val_accuracy", save_best_only=True),
]

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks_ft
)

In [ ]:
val_loss, val_acc = model.evaluate(val_gen)
print(f"Final validation accuracy: {val_acc*100:.2f}%")

In [ ]:
def plot_history(h1, h2):
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]

    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(acc, label="train"); plt.plot(val_acc, label="val")
    plt.axvline(len(h1.history["accuracy"]), color="gray", linestyle="--", label="fine-tune start")
    plt.title("Accuracy"); plt.legend()

    plt.subplot(1,2,2)
    plt.plot(loss, label="train"); plt.plot(val_loss, label="val")
    plt.axvline(len(h1.history["loss"]), color="gray", linestyle="--")
    plt.title("Loss"); plt.legend()
    plt.show()

plot_history(history1, history2)

In [ ]:
recycling_advice = {
    "cardboard": "Recyclable — flatten and place in paper/cardboard recycling.",
    "glass": "Recyclable — rinse and place in glass recycling bin.",
    "metal": "Recyclable — rinse cans, place in metal recycling.",
    "paper": "Recyclable — keep dry, place in paper recycling.",
    "plastic": "Check resin code; most PET/HDPE plastics are recyclable.",
    "trash": "Not recyclable — dispose in general waste.",
}

def classify_and_recommend(img_path):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
    arr = tf.keras.preprocessing.image.img_to_array(img)
    arr = preprocess_input(arr)
    arr = np.expand_dims(arr, axis=0)

    pred = model.predict(arr)[0]
    idx = np.argmax(pred)
    label = class_names[idx]
    confidence = pred[idx] * 100

    print(f"Predicted: {label} ({confidence:.1f}% confidence)")
    print(f"Recommendation: {recycling_advice.get(label, 'No advice available.')}")
    return label, confidence

# Example:
# classify_and_recommend("garbage_split/val/plastic/plastic123.jpg")

In [ ]:
# Save in native Keras format (recommended)
model.save("garbage_classifier_efficientnetb3.keras")

# Also save just the weights (smaller, useful if you rebuild the architecture later)
model.save_weights("garbage_classifier_efficientnetb3.weights.h5")

print("Saved successfully.")

In [ ]:
import json

with open("class_indices.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print(train_gen.class_indices)

In [ ]:
from google.colab import files

files.download("garbage_classifier_efficientnetb3.keras")
files.download("class_indices.json")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("garbage_classifier_efficientnetb3.keras", "/content/drive/MyDrive/garbage_classifier_efficientnetb3.keras")
shutil.copy("class_indices.json", "/content/drive/MyDrive/class_indices.json")

print("Saved to Google Drive.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder in Drive for this project
!mkdir -p /content/drive/MyDrive/garbage_classification

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/garbage_classification/data"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.lower().endswith(".zip"):
            print(os.path.join(root, file))

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/garbage classification project/dataset/archive.zip"

extract_path = "/content/drive/MyDrive/garbage classification project/dataset/extracted"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Extraction completed successfully!")
print(os.listdir(extract_path))

In [ ]:
import os

for root, dirs, files in os.walk(extract_path):
    if dirs:
        print(root)
        print(dirs)

In [ ]:
dataset_path = "/content/drive/MyDrive/garbage classification project/dataset/extracted/Garbage classification/Garbage classification"

In [ ]:
import os

print(os.listdir(dataset_path))

In [ ]:
import os

# Check if you previously saved the split folder to Drive
possible_split = "/content/drive/MyDrive/garbage_classification/garbage_split"

if os.path.exists(possible_split):
    print("Found existing split at:", possible_split)
    print(os.listdir(possible_split))
else:
    print("No existing split found — you'll need to recreate it (see next cell).")

In [ ]:
import os

DATA_DIR = "/content/drive/MyDrive/garbage classification project/dataset/extracted/Garbage classification/Garbage classification"

BASE_DIR = "garbage_split"

print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("BASE_DIR exists:", os.path.exists(BASE_DIR))

In [ ]:
import os
import shutil
import random

BASE_DIR = "garbage_split"

if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)

classes = [
    c for c in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, c))
]

val_ratio = 0.2

random.seed(42)

for cls in classes:
    src = os.path.join(DATA_DIR, cls)

    imgs = [
        f for f in os.listdir(src)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(imgs)

    split = int(len(imgs) * (1 - val_ratio))

    for phase, subset in [
        ("train", imgs[:split]),
        ("val", imgs[split:])
    ]:
        dst = os.path.join(BASE_DIR, phase, cls)
        os.makedirs(dst, exist_ok=True)

        for img in subset:
            shutil.copy(
                os.path.join(src, img),
                os.path.join(dst, img)
            )

print("Classes:", classes)

print(
    "Train counts:",
    {c: len(os.listdir(f"{BASE_DIR}/train/{c}")) for c in classes}
)

print(
    "Val counts:",
    {c: len(os.listdir(f"{BASE_DIR}/val/{c}")) for c in classes}
)

In [ ]:
f"{BASE_DIR}/train"
f"{BASE_DIR}/val"

In [ ]:
import os

print(os.listdir("garbage_split/val"))

In [ ]:
print("BASE_DIR:", BASE_DIR)
print("Train:", os.listdir("garbage_split/train"))
print("Val:", os.listdir("garbage_split/val"))

In [ ]:
import numpy as np
import json
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
IMG_SIZE_R = (224, 224)
BATCH_SIZE = 32

train_datagen_r = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
)

val_datagen_r = ImageDataGenerator(preprocessing_function=resnet_preprocess)

train_gen_r = train_datagen_r.flow_from_directory(
    f"{BASE_DIR}/train", target_size=IMG_SIZE_R, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True
)
val_gen_r = val_datagen_r.flow_from_directory(
    f"{BASE_DIR}/val", target_size=IMG_SIZE_R, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

NUM_CLASSES = train_gen_r.num_classes
class_names_r = list(train_gen_r.class_indices.keys())
print(train_gen_r.class_indices)

In [ ]:
y_train_r = train_gen_r.classes
class_weights_r = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_r),
    y=y_train_r
)
class_weight_dict_r = dict(enumerate(class_weights_r))
print(class_weight_dict_r)

In [ ]:
base_model_r = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE_R, 3)
)
base_model_r.trainable = False

x = base_model_r.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

resnet_model = Model(inputs=base_model_r.input, outputs=outputs)
resnet_model.summary()

In [ ]:
resnet_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_r1 = [
    EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("best_resnet_stage1.keras", monitor="val_accuracy", save_best_only=True),
]

history_r1 = resnet_model.fit(
    train_gen_r,
    validation_data=val_gen_r,
    epochs=20,
    class_weight=class_weight_dict_r,
    callbacks=callbacks_r1
)

In [ ]:
base_model_r.trainable = True
fine_tune_at_r = int(len(base_model_r.layers) * 0.3)  # unfreeze last 70%
for layer in base_model_r.layers[:fine_tune_at_r]:
    layer.trainable = False

resnet_model.compile(
    optimizer=Adam(learning_rate=5e-6),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_r2 = [
    EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint("best_resnet_stage2.keras", monitor="val_accuracy", save_best_only=True),
]

history_r2 = resnet_model.fit(
    train_gen_r,
    validation_data=val_gen_r,
    epochs=30,
    class_weight=class_weight_dict_r,
    callbacks=callbacks_r2
)

In [ ]:
base_model_r.trainable = True  # unfreeze everything

resnet_model.compile(
    optimizer=Adam(learning_rate=1e-6),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_r3 = [
    EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    ModelCheckpoint("best_resnet_final.keras", monitor="val_accuracy", save_best_only=True),
]

history_r3 = resnet_model.fit(
    train_gen_r,
    validation_data=val_gen_r,
    epochs=15,
    class_weight=class_weight_dict_r,
    callbacks=callbacks_r3
)

In [ ]:
val_loss_r, val_acc_r = resnet_model.evaluate(val_gen_r)
print(f"ResNet50 final validation accuracy: {val_acc_r*100:.2f}%")

In [ ]:
resnet_model.save("garbage_classifier_resnet50.keras")

with open("class_indices_resnet.json", "w") as f:
    json.dump(train_gen_r.class_indices, f)

from google.colab import files
files.download("garbage_classifier_resnet50.keras")
files.download("class_indices_resnet.json")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

val_gen_r.reset()
preds_r = resnet_model.predict(val_gen_r)
y_pred_r = np.argmax(preds_r, axis=1)
y_true_r = val_gen_r.classes

print(classification_report(y_true_r, y_pred_r, target_names=class_names_r))

cm_r = confusion_matrix(y_true_r, y_pred_r)
plt.figure(figsize=(7,6))
sns.heatmap(cm_r, annot=True, fmt="d", xticklabels=class_names_r, yticklabels=class_names_r, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("ResNet50 Confusion Matrix")
plt.show()

In [ ]:
DATA_DIR = "/content/drive/MyDrive/garbage classification project/dataset/extracted/Garbage classification/Garbage classification"

In [ ]:
import os

print(os.listdir(DATA_DIR))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from PIL import Image

from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

BASE_DIR = "/content/drive/MyDrive/garbage_classification/garbage_split"
print("Classes:", os.listdir(f"{BASE_DIR}/train"))

In [ ]:
valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
removed = []

for phase in ["train", "val"]:
    for cls in os.listdir(f"{BASE_DIR}/{phase}"):
        folder = f"{BASE_DIR}/{phase}/{cls}"
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            ext = os.path.splitext(fname)[1].lower()

            if ext not in valid_extensions:
                os.remove(fpath)
                removed.append(fpath)
                continue

            try:
                img = Image.open(fpath)
                img.verify()
            except Exception:
                os.remove(fpath)
                removed.append(fpath)

print(f"Removed {len(removed)} bad/non-image files.")
for f in removed:
    print(" -", f)

In [ ]:
def mobilenet_manual_preprocess(img):
    img = img / 127.5
    img = img - 1.0
    return img

IMG_SIZE_M = (224, 224)
BATCH_SIZE = 32

train_datagen_m = ImageDataGenerator(
    preprocessing_function=mobilenet_manual_preprocess,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
)

val_datagen_m = ImageDataGenerator(preprocessing_function=mobilenet_manual_preprocess)

train_gen_m = train_datagen_m.flow_from_directory(
    f"{BASE_DIR}/train", target_size=IMG_SIZE_M, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=True
)
val_gen_m = val_datagen_m.flow_from_directory(
    f"{BASE_DIR}/val", target_size=IMG_SIZE_M, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

NUM_CLASSES = train_gen_m.num_classes
class_names_m = list(train_gen_m.class_indices.keys())
print("Class indices:", train_gen_m.class_indices)

# Sanity check
batch_x, batch_y = next(train_gen_m)
print("Pixel range:", batch_x.min(), "to", batch_x.max())

In [ ]:
y_train_m = train_gen_m.classes
class_weights_m = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train_m), y=y_train_m
)
class_weight_dict_m = dict(enumerate(class_weights_m))
print(class_weight_dict_m)

In [ ]:
base_model_m = MobileNetV3Large(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE_M, 3),
    include_preprocessing=False
)
base_model_m.trainable = False

x = base_model_m.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation="softmax")(x)

mobilenet_model = Model(inputs=base_model_m.input, outputs=outputs)
mobilenet_model.summary()

In [ ]:
mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

callbacks_m1 = [
    EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint("best_mobilenet_stage1.keras", monitor="val_accuracy", save_best_only=True),
]

history_m1 = mobilenet_model.fit(
    train_gen_m,
    validation_data=val_gen_m,
    epochs=20,
    class_weight=class_weight_dict_m,
    callbacks=callbacks_m1
)

In [ ]:
base_model_m.trainable = True
fine_tune_at_m = int(len(base_model_m.layers) * 0.3)
for layer in base_model_m.layers[:fine_tune_at_m]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=Adam(learning_rate=5e-6),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

callbacks_m2 = [
    EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint("best_mobilenet_stage2.keras", monitor="val_accuracy", save_best_only=True),
]

history_m2 = mobilenet_model.fit(
    train_gen_m,
    validation_data=val_gen_m,
    epochs=30,
    class_weight=class_weight_dict_m,
    callbacks=callbacks_m2
)

In [ ]:
base_model_m.trainable = True

mobilenet_model.compile(
    optimizer=Adam(learning_rate=1e-6),
    loss=CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

callbacks_m3 = [
    EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    ModelCheckpoint("best_mobilenet_final.keras", monitor="val_accuracy", save_best_only=True),
]

history_m3 = mobilenet_model.fit(
    train_gen_m,
    validation_data=val_gen_m,
    epochs=15,
    class_weight=class_weight_dict_m,
    callbacks=callbacks_m3
)

In [ ]:
val_loss_m, val_acc_m = mobilenet_model.evaluate(val_gen_m)
print(f"MobileNetV3 final validation accuracy: {val_acc_m*100:.2f}%")

In [ ]:
save_dir = "/content/drive/MyDrive/garbage_classification/models"
os.makedirs(save_dir, exist_ok=True)

mobilenet_model.save(f"{save_dir}/garbage_classifier_mobilenetv3.keras")

with open(f"{save_dir}/class_indices_mobilenet.json", "w") as f:
    json.dump(train_gen_m.class_indices, f)

print("Saved to:", save_dir)

In [ ]:
from google.colab import files

files.download(f"{save_dir}/garbage_classifier_mobilenetv3.keras")
files.download(f"{save_dir}/class_indices_mobilenet.json")

In [ ]:
val_gen_m.reset()
preds_m = mobilenet_model.predict(val_gen_m)
y_pred_m = np.argmax(preds_m, axis=1)
y_true_m = val_gen_m.classes

print(classification_report(y_true_m, y_pred_m, target_names=class_names_m))

cm_m = confusion_matrix(y_true_m, y_pred_m)
plt.figure(figsize=(7,6))
sns.heatmap(cm_m, annot=True, fmt="d", xticklabels=class_names_m, yticklabels=class_names_m, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("MobileNetV3 Confusion Matrix")
plt.show()

In [ ]:
import pandas as pd

accuracies = {
    "EfficientNet-B3": 89.37,
    "ResNet50": 80.00,
    "MobileNetV3": 90.35
}

comparison = pd.DataFrame(
    list(accuracies.items()),
    columns=["Model", "Validation Accuracy (%)"]
)

print(comparison)

best_model = max(accuracies, key=accuracies.get)
best_accuracy = accuracies[best_model]

print("\nBest Model:", best_model)
print("Best Validation Accuracy:", best_accuracy, "%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.bar(
    comparison["Model"],
    comparison["Validation Accuracy (%)"]
)

plt.xlabel("Model")
plt.ylabel("Validation Accuracy (%)")
plt.title("Garbage Classification Model Comparison")
plt.ylim(0, 100)

plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, random

# Point back to your original extracted data (before train/val split)
DATA_DIR = None
expected_classes = {"cardboard", "glass", "metal", "paper", "plastic", "trash"}
search_root = "/content/drive/MyDrive/garbage_classification/data"

for root, dirs, files_ in os.walk(search_root):
    if expected_classes.issubset(set(d.lower() for d in dirs)):
        DATA_DIR = root
        break

print("Source data found at:", DATA_DIR)

TEST_BASE_DIR = "/content/drive/MyDrive/garbage_classification/garbage_split_with_test"
if os.path.exists(TEST_BASE_DIR):
    shutil.rmtree(TEST_BASE_DIR)

classes = [c for c in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, c))]
train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15

for cls in classes:
    src = os.path.join(DATA_DIR, cls)
    imgs = [f for f in os.listdir(src) if not f.startswith(".")]
    random.seed(42)  # reproducible split
    random.shuffle(imgs)

    n = len(imgs)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)

    splits = {
        "train": imgs[:train_end],
        "val": imgs[train_end:val_end],
        "test": imgs[val_end:]
    }

    for phase, subset in splits.items():
        dst = os.path.join(TEST_BASE_DIR, phase, cls)
        os.makedirs(dst, exist_ok=True)
        for img in subset:
            shutil.copy(os.path.join(src, img), os.path.join(dst, img))

print("Split created.")
for phase in ["train", "val", "test"]:
    print(f"\n{phase.upper()} counts:")
    print({c: len(os.listdir(f"{TEST_BASE_DIR}/{phase}/{c}")) for c in classes})

In [ ]:

import os

BASE_DIR = "/content/drive/MyDrive/garbage_classification/garbage_split"
print(os.listdir(f"{BASE_DIR}/train"))

In [ ]:
print(os.listdir("/content/drive/MyDrive/garbage_classification"))

In [ ]:
import os
from PIL import Image


valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
removed = []

for phase in ["train", "val"]:
    for cls in os.listdir(f"{BASE_DIR}/{phase}"):
        folder = f"{BASE_DIR}/{phase}/{cls}"
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            ext = os.path.splitext(fname)[1].lower()

            if ext not in valid_extensions:
                os.remove(fpath)
                removed.append(fpath)
                continue

            try:
                img = Image.open(fpath)
                img.verify()
            except Exception:
                os.remove(fpath)
                removed.append(fpath)

print(f"Removed {len(removed)} bad/non-image files.")
for f in removed:
    print(" -", f)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

def mobilenet_manual_preprocess(img):
    return (img / 127.5) - 1.0

BATCH_SIZE = 32

test_gen_e = ImageDataGenerator(preprocessing_function=efficientnet_preprocess).flow_from_directory(
    f"{TEST_BASE_DIR}/test", target_size=(300, 300), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen_r = ImageDataGenerator(preprocessing_function=resnet_preprocess).flow_from_directory(
    f"{TEST_BASE_DIR}/test", target_size=(224, 224), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen_m = ImageDataGenerator(preprocessing_function=mobilenet_manual_preprocess).flow_from_directory(
    f"{TEST_BASE_DIR}/test", target_size=(224, 224), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

class_names = list(test_gen_e.class_indices.keys())
print("Classes:", class_names)

In [ ]:
from tensorflow.keras.models import load_model
import os

# Correct paths based on where your models were actually saved
efficientnet_path = "/content/drive/MyDrive/garbage_classifier_efficientnetb3.keras"
resnet_path = "/content/drive/MyDrive/garbage_classifier_resnet50.keras"
mobilenet_path = "/content/drive/MyDrive/garbage_classification/models/garbage_classifier_mobilenetv3.keras"

print("EfficientNet exists:", os.path.exists(efficientnet_path))
print("ResNet50 exists:", os.path.exists(resnet_path))
print("MobileNetV3 exists:", os.path.exists(mobilenet_path))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = "/content/drive/MyDrive/garbage_classification/models"
print(os.listdir(save_dir))

In [ ]:
from google.colab import files
uploaded = files.upload()

import shutil, os

save_dir = "/content/drive/MyDrive/garbage_classification/models"
os.makedirs(save_dir, exist_ok=True)

for fname in uploaded.keys():
    shutil.move(fname, f"{save_dir}/{fname}")
    print(f"Moved {fname} to Drive.")

In [ ]:
import os

save_dir = "/content/drive/MyDrive/garbage_classification/models"
print(os.listdir(save_dir))

In [ ]:
from tensorflow.keras.models import load_model

save_dir = "/content/drive/MyDrive/garbage_classification/models"

resnet_model = load_model(f"{save_dir}/garbage_classifier_resnet50.keras")
print("ResNet50 loaded successfully.")

In [ ]:
import json

with open(f"{save_dir}/class_indices_resnet.json") as f:
    class_indices_resnet = json.load(f)

print(class_indices_resnet)

In [ ]:
from tensorflow.keras.models import load_model
import os

efficientnet_path = "/content/drive/MyDrive/garbage_classifier_efficientnetb3.keras"
resnet_path = "/content/drive/MyDrive/garbage_classification/models/garbage_classifier_resnet50.keras"
mobilenet_path = "/content/drive/MyDrive/garbage_classification/models/garbage_classifier_mobilenetv3.keras"

print("EfficientNet exists:", os.path.exists(efficientnet_path))
print("ResNet50 exists:", os.path.exists(resnet_path))
print("MobileNetV3 exists:", os.path.exists(mobilenet_path))

In [ ]:
from tensorflow.keras.models import load_model

MODEL_DIR = "/content/drive/MyDrive/garbage_classification/models"

efficientnet_model = load_model(f"{MODEL_DIR}/garbage_classifier_efficientnetb3.keras")
resnet_model = load_model(f"{MODEL_DIR}/garbage_classifier_resnet50.keras")
mobilenet_model = load_model(f"{MODEL_DIR}/garbage_classifier_mobilenetv3.keras")

print("All three models loaded successfully.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.metrics import classification_report, confusion_matrix

def mobilenet_manual_preprocess(img):
    return (img / 127.5) - 1.0

MODEL_DIR = "/content/drive/MyDrive/garbage_classification/models"
print("Models available:", os.listdir(MODEL_DIR))

In [ ]:
efficientnet_model = load_model(f"{MODEL_DIR}/garbage_classifier_efficientnetb3.keras")
resnet_model = load_model(f"{MODEL_DIR}/garbage_classifier_resnet50.keras")
mobilenet_model = load_model(f"{MODEL_DIR}/garbage_classifier_mobilenetv3.keras")

print("All three models loaded successfully.")

In [ ]:
import shutil, random

TEST_BASE_DIR = "/content/drive/MyDrive/garbage_classification/garbage_split_with_test"

if os.path.exists(f"{TEST_BASE_DIR}/test"):
    print("Test split already exists. Using it directly.")
else:
    print("Creating test split...")
    DATA_DIR = None
    expected_classes = {"cardboard", "glass", "metal", "paper", "plastic", "trash"}
    search_root = "/content/drive/MyDrive/garbage_classification/data"

    for root, dirs, files_ in os.walk(search_root):
        if expected_classes.issubset(set(d.lower() for d in dirs)):
            DATA_DIR = root
            break

    classes = [c for c in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, c))]
    train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15

    for cls in classes:
        src = os.path.join(DATA_DIR, cls)
        imgs = [f for f in os.listdir(src) if not f.startswith(".")]
        random.seed(42)
        random.shuffle(imgs)

        n = len(imgs)
        train_end = int(n * train_ratio)
        val_end = train_end + int(n * val_ratio)

        splits = {"train": imgs[:train_end], "val": imgs[train_end:val_end], "test": imgs[val_end:]}

        for phase, subset in splits.items():
            dst = os.path.join(TEST_BASE_DIR, phase, cls)
            os.makedirs(dst, exist_ok=True)
            for img in subset:
                shutil.copy(os.path.join(src, img), os.path.join(dst, img))

    print("Test split created.")

classes = sorted(os.listdir(f"{TEST_BASE_DIR}/test"))
print("Classes:", classes)
print("Test counts:", {c: len(os.listdir(f"{TEST_BASE_DIR}/test/{c}")) for c in classes})

In [ ]:
BATCH_SIZE = 64

test_gen_e = ImageDataGenerator(preprocessing_function=efficientnet_preprocess).flow_from_directory(
    f"{TEST_BASE_DIR}/test", target_size=(300, 300), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen_r = ImageDataGenerator(preprocessing_function=resnet_preprocess).flow_from_directory(
    f"{TEST_BASE_DIR}/test", target_size=(224, 224), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen_m = ImageDataGenerator(preprocessing_function=mobilenet_manual_preprocess).flow_from_directory(
    f"{TEST_BASE_DIR}/test", target_size=(224, 224), batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

class_names = list(test_gen_e.class_indices.keys())
print("Class indices:", test_gen_e.class_indices)

In [ ]:
print("Testing EfficientNetB3...")
test_loss_e, test_acc_e = efficientnet_model.evaluate(test_gen_e, verbose=0)

print("Testing ResNet50...")
test_loss_r, test_acc_r = resnet_model.evaluate(test_gen_r, verbose=0)

print("Testing MobileNetV3...")
test_loss_m, test_acc_m = mobilenet_model.evaluate(test_gen_m, verbose=0)

print(f"\nEfficientNetB3 TEST accuracy: {test_acc_e*100:.2f}%")
print(f"ResNet50 TEST accuracy:       {test_acc_r*100:.2f}%")
print(f"MobileNetV3 TEST accuracy:    {test_acc_m*100:.2f}%")

In [ ]:
def evaluate_and_plot(model, test_gen, model_name):
    test_gen.reset()
    preds = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes

    print(f"\n{'='*20} {model_name} TEST REPORT {'='*20}")
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
    plt.title(f"{model_name} — Test Set Confusion Matrix")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

evaluate_and_plot(efficientnet_model, test_gen_e, "EfficientNetB3")
evaluate_and_plot(resnet_model, test_gen_r, "ResNet50")
evaluate_and_plot(mobilenet_model, test_gen_m, "MobileNetV3")

In [ ]:
test_results_df = pd.DataFrame({
    "Model": ["EfficientNetB3", "ResNet50", "MobileNetV3"],
    "Test Accuracy (%)": [test_acc_e*100, test_acc_r*100, test_acc_m*100],
    "Test Loss": [test_loss_e, test_loss_r, test_loss_m]
}).sort_values("Test Accuracy (%)", ascending=False).reset_index(drop=True)

print(test_results_df.to_string(index=False))

test_results_df.to_csv("/content/drive/MyDrive/garbage_classification/test_results.csv", index=False)
print("\nSaved test_results.csv to Drive.")

In [ ]:
recycling_advice = {
    "cardboard": "Recyclable — flatten and place in paper/cardboard recycling.",
    "glass": "Recyclable — rinse and place in glass recycling bin.",
    "metal": "Recyclable — rinse cans, place in metal recycling.",
    "paper": "Recyclable — keep dry, place in paper recycling.",
    "plastic": "Check resin code; most PET/HDPE plastics are recyclable.",
    "trash": "Not recyclable — dispose in general waste.",
}

def classify_and_recommend(img_path, model, preprocess_fn, img_size, model_name="Model"):
    img = load_img(img_path, target_size=img_size)
    arr = img_to_array(img)
    arr = preprocess_fn(arr)
    arr = np.expand_dims(arr, axis=0)

    pred = model.predict(arr, verbose=0)[0]
    idx = np.argmax(pred)
    label = class_names[idx]
    confidence = pred[idx] * 100

    print(f"[{model_name}] Predicted: {label} ({confidence:.1f}% confidence)")
    print(f"Recommendation: {recycling_advice.get(label, 'No advice available.')}")
    return label, confidence

# Example — test on a sample from the test folder (EfficientNetB3)
sample_path = f"{TEST_BASE_DIR}/test/plastic/" + os.listdir(f"{TEST_BASE_DIR}/test/plastic")[0]
classify_and_recommend(sample_path, efficientnet_model, efficientnet_preprocess, (300,300), "EfficientNetB3")

In [ ]:
num_images = 6
results = []

for i in range(num_images):
    print(f"\n{'='*15} Upload photo {i+1} of {num_images} {'='*15}")
    uploaded = files.upload()
    img_path = list(uploaded.keys())[0]

    print(f"\nTesting: {img_path}")
    label, confidence = classify_and_recommend(img_path, efficientnet_model, efficientnet_preprocess, (300,300), "EfficientNetB3")
    results.append((img_path, label, confidence))

print(f"\n{'='*20} SUMMARY {'='*20}")
for fname, label, conf in results:
    print(f"{fname} → {label} ({conf:.1f}%)")

In [ ]:
actual_labels = ["cardboard", "metal", "glass", "paper", "plastic", "trash"]
predicted_labels = [label for (fname, label, conf) in results]

correct = sum([1 for a, p in zip(actual_labels, predicted_labels) if a == p])
total = len(actual_labels)
real_world_accuracy = (correct / total) * 100

print(f"Real-world test: {correct}/{total} correct")
print(f"Real-world accuracy: {real_world_accuracy:.2f}%")

print("\nDetailed comparison:")
for a, p, (fname, label, conf) in zip(actual_labels, predicted_labels, results):
    status = "✅" if a == p else "❌"
    print(f"{status} Actual: {a:<12} Predicted: {p:<12} Confidence: {conf:.1f}%")

In [ ]:
print("="*55)
print("FINAL MODEL PERFORMANCE SUMMARY — EfficientNetB3")
print("="*55)
print(f"Test Set Accuracy (Kaggle held-out):  {test_acc_e*100:.2f}%")
print(f"Real-World Accuracy (6 new photos):   {real_world_accuracy:.2f}%")

In [ ]:
final_summary = """FINAL MODEL PERFORMANCE SUMMARY — EfficientNetB3

Test Set Accuracy (Kaggle held-out): 96.61%
Real-World Accuracy (6 new photos): 83.33%

Final Model: EfficientNet-B3
"""

save_path = "/content/drive/MyDrive/garbage_classification/final_model_performance.txt"

with open(save_path, "w") as f:
    f.write(final_summary)

print("Saved successfully!")
print(save_path)

In [ ]:
import pandas as pd

summary_df = pd.DataFrame({
    "Metric": [
        "Test Set Accuracy (Kaggle held-out)",
        "Real-World Accuracy (6 new photos)"
    ],
    "Accuracy (%)": [
        96.61,
        83.33
    ]
})

save_path = "/content/drive/MyDrive/garbage_classification/final_model_performance.csv"

summary_df.to_csv(save_path, index=False)

print("Saved successfully!")
print(save_path)

In [ ]:
!pip install streamlit --quiet
!pip install pyngrok --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
app_code = '''
import streamlit as st
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image

st.set_page_config(page_title="Smart Garbage Classifier", page_icon="♻️", layout="centered")

@st.cache_resource
def load_my_model():
    MODEL_DIR = "/content/drive/MyDrive/garbage_classification/models"
    return load_model(f"{MODEL_DIR}/garbage_classifier_efficientnetb3.keras")

model = load_my_model()

class_names = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

recycling_advice = {
    "cardboard": "Recyclable — flatten and place in paper/cardboard recycling.",
    "glass": "Recyclable — rinse and place in glass recycling bin.",
    "metal": "Recyclable — rinse cans, place in metal recycling.",
    "paper": "Recyclable — keep dry, place in paper recycling.",
    "plastic": "Check resin code; most PET/HDPE plastics are recyclable.",
    "trash": "Not recyclable — dispose in general waste.",
}

st.title("♻️ Smart Garbage Classification & Recycling Recommendation Agent")
st.write("Upload a photo of garbage/trash to get instant classification and recycling advice.")

uploaded_file = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png", "webp"])

if uploaded_file is not None:
    img = Image.open(uploaded_file).convert("RGB")
    st.image(img, caption="Uploaded Image", use_container_width=True)

    img_resized = img.resize((300, 300))
    arr = img_to_array(img_resized)
    arr = efficientnet_preprocess(arr)
    arr = np.expand_dims(arr, axis=0)

    pred = model.predict(arr, verbose=0)[0]
    idx = np.argmax(pred)
    label = class_names[idx]
    confidence = pred[idx] * 100

    st.subheader(f"Prediction: {label.upper()}")
    st.write(f"Confidence: {confidence:.1f}%")
    st.success(recycling_advice.get(label, "No advice available."))

    st.subheader("Confidence per class")
    probs_dict = {class_names[i]: float(pred[i]) for i in range(len(class_names))}
    st.bar_chart(probs_dict)
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py file updated.")

In [ ]:
from pyngrok import ngrok
NGROK_AUTH_TOKEN = "3Ho9Ffb73YkJig1Xl3OHouxEP7C_4bLxkN2sLF58df4gkoJuW"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

from pyngrok import ngrok
NGROK_AUTH_TOKEN = "3Ho9Ffb73YkJig1Xl3OHouxEP7C_4bLxkN2sLF58df4gkoJuW"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("Token set successfully.")

In [ ]:
import subprocess, time
from pyngrok import ngrok
ngrok.kill()
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(5)

public_url = ngrok.connect(8501)
print("Your Streamlit app is live at:", public_url)

In [ ]:
import shutil

shutil.copy("app.py", "/content/drive/MyDrive/garbage_classification/app.py")
print("app.py saved to Drive.")

In [ ]:
import shutil
shutil.copy("/content/drive/MyDrive/garbage_classification/app.py", "app.py")